# Feature Engineering reutilizable

Este notebook explica cómo los datos originales se convierten en una matriz que el modelo puede utilizar. Las transformaciones no se redefinen aquí: se importa la misma función de `src/features.py` utilizada durante entrenamiento e inferencia.

## 1. Objetivo de la preparación de características

El dataset mezcla números, categorías y valores faltantes. El modelo necesita una representación numérica consistente. La preparación debe aprenderse con entrenamiento y aplicarse sin cambios a prueba o producción.

El flujo es:

```text
Datos originales → separación train/test → ajuste con train → transformación de train y test
```

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from sklearn.model_selection import train_test_split

from src.config import RANDOM_STATE, TEST_SIZE
from src.data import load_adult, split_features_target
from src.features import build_feature_engineering

df = load_adult()
X, y = split_features_target(df)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


Entrenamiento: (39073, 14)
Prueba: (9769, 14)


## 2. Separación antes de aprender transformaciones

La división se realiza antes de calcular medianas, modas o categorías. De esta forma, el conjunto de prueba representa información no vista y ofrece una evaluación más honesta.

La estratificación mantiene en ambos conjuntos una proporción similar de personas con ingresos altos y bajos.

In [2]:
class_balance = {
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}
class_balance


{'train': income
 0    0.76073
 1    0.23927
 Name: proportion, dtype: float64,
 'test': income
 0    0.760672
 1    0.239328
 Name: proportion, dtype: float64}

## 3. Tratamiento de variables numéricas

Las variables numéricas se completan con la mediana aprendida en entrenamiento. También se crea un indicador que informa al modelo cuándo un valor estaba ausente. Finalmente se estandarizan para que la definición de características pueda reutilizarse con otros modelos.

La mediana se prefiere a la media porque variables como capital y peso censal tienen distribuciones muy asimétricas.

## 4. Tratamiento de variables categóricas

Los faltantes categóricos se completan con la categoría más frecuente aprendida en entrenamiento. Después, cada categoría se convierte en columnas binarias mediante one-hot encoding.

Las categorías nuevas se ignoran de forma segura durante inferencia. Esto evita que una categoría no observada durante entrenamiento detenga una predicción.

## 5. Ajuste exclusivo con entrenamiento

En esta etapa se aprenden medianas, modas y categorías usando solamente `X_train`. Luego esas mismas reglas se aplican a `X_test`, sin recalcularlas. Esta separación evita leakage y reproduce el comportamiento que tendrá el modelo en producción.

In [3]:
features = build_feature_engineering()
X_train_transformed = features.fit_transform(X_train)
X_test_transformed = features.transform(X_test)

print("Entrada original de train:", X_train.shape)
print("Train transformado:", X_train_transformed.shape)
print("Test transformado:", X_test_transformed.shape)


Entrada original de train: (39073, 14)
Train transformado: (39073, 105)
Test transformado: (9769, 105)


## 6. Resultado de la transformación

Las 14 variables originales producen más columnas porque cada categoría del one-hot encoding se convierte en una característica independiente. Train y test deben terminar con exactamente el mismo número y orden de columnas.

In [4]:
feature_names = features.get_feature_names_out()
print(f"Cantidad final de características: {len(feature_names)}")
feature_names[:30]


Cantidad final de características: 105


array(['numeric__age', 'numeric__fnlwgt', 'numeric__education-num',
       'numeric__capital-gain', 'numeric__capital-loss',
       'numeric__hours-per-week', 'categorical__workclass_Federal-gov',
       'categorical__workclass_Local-gov',
       'categorical__workclass_Never-worked',
       'categorical__workclass_Private',
       'categorical__workclass_Self-emp-inc',
       'categorical__workclass_Self-emp-not-inc',
       'categorical__workclass_State-gov',
       'categorical__workclass_Without-pay',
       'categorical__education_10th', 'categorical__education_11th',
       'categorical__education_12th', 'categorical__education_1st-4th',
       'categorical__education_5th-6th', 'categorical__education_7th-8th',
       'categorical__education_9th', 'categorical__education_Assoc-acdm',
       'categorical__education_Assoc-voc',
       'categorical__education_Bachelors',
       'categorical__education_Doctorate',
       'categorical__education_HS-grad', 'categorical__education_Maste

## 7. Inspección de las reglas aprendidas

El transformador conserva dentro de sí las estadísticas obtenidas de entrenamiento. Esto permite comprobar qué medianas y categorías se utilizarán después, y garantiza que viajen junto con el modelo guardado.

In [5]:
numeric_pipeline = features.named_transformers_["numeric"]
categorical_pipeline = features.named_transformers_["categorical"]

print("Medianas aprendidas:", numeric_pipeline.named_steps["imputer"].statistics_)
print("Cantidad de categorías por variable:", [
    len(categories)
    for categories in categorical_pipeline.named_steps["one_hot"].categories_
])


Medianas aprendidas: [3.70000e+01 1.78615e+05 1.00000e+01 0.00000e+00 0.00000e+00 4.00000e+01]
Cantidad de categorías por variable: [8, 16, 7, 14, 6, 5, 2, 41]


## 8. Conclusión

El feature engineering resuelve faltantes, convierte categorías y mantiene una representación estable entre entrenamiento, evaluación e inferencia. Al integrarlo con el modelo en un único pipeline, una predicción futura puede recibir las columnas originales sin repetir manualmente ninguna preparación.